# Sección 5 – N-HiTS y TCN con NeuralForecast (Nixtla)

Este notebook entrena y evalúa dos modelos de deep-learning para forecasting univariado de series de consumo eléctrico (dataset UCI ElectricityLoadDiagrams):

| Modelo | Descripción |
|--------|-------------|
| **N-HiTS** | Neural Hierarchical Interpolation for Time Series |
| **TCN** | Temporal Convolutional Network |

**Protocolo:** Backtest rolling-origin con 92 orígenes (oct–dic 2014), horizonte H = 24 h.

## 0. Configuración e Imports

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Agregar raíz del proyecto al path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# NeuralForecast
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS, TCN

# Proyecto
from src.backtest import backtest, ForecastModel
from src.metrics import evaluate_all, mae_per_horizon
from src.utils import set_seed

set_seed(42)
print('Imports OK')

## 1. Carga de Datos

In [ ]:
# ── Cargar panel wide (clientes × tiempo) ────────────────────────────────────
DATA_PATH = os.path.join(ROOT, 'data', 'processed')

# Intentar cargar panel procesado; si no existe, descargar y procesar
panel_file = os.path.join(DATA_PATH, 'panel.parquet')
if not os.path.exists(panel_file):
    panel_file = os.path.join(DATA_PATH, 'panel.csv')

if os.path.exists(panel_file):
    if panel_file.endswith('.parquet'):
        panel = pd.read_parquet(panel_file, parse_dates=['ds'] if 'ds' in pd.read_parquet(panel_file).columns else True)
    else:
        panel = pd.read_csv(panel_file, index_col=0, parse_dates=True)
    print(f'Panel cargado desde {panel_file}: {panel.shape}')
else:
    print('Panel no encontrado — descargando y procesando datos UCI...')
    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00321/LD2011_2014.txt.zip'
    raw = pd.read_csv(url, sep=';', decimal=',', index_col=0, parse_dates=True)
    raw.index = pd.to_datetime(raw.index)
    raw = raw.resample('H').mean()  # 15 min → 1 h
    raw = raw.dropna(axis=1, how='all')
    # Seleccionar primeros 20 clientes para velocidad
    panel = raw.iloc[:, :20]
    os.makedirs(DATA_PATH, exist_ok=True)
    panel.to_parquet(panel_file)
    print(f'Panel guardado: {panel.shape}')

# Asegurar índice datetime
if not isinstance(panel.index, pd.DatetimeIndex):
    panel.index = pd.to_datetime(panel.index)

panel = panel.sort_index()
print(f'Rango: {panel.index.min()} → {panel.index.max()}')
print(f'Clientes: {panel.shape[1]}, Pasos de tiempo: {panel.shape[0]}')

In [ ]:
# ── Splits temporales ─────────────────────────────────────────────────────────
TRAIN_END  = '2014-07-31 23:00:00'
VAL_END    = '2014-09-30 23:00:00'
TEST_START = '2014-10-01'
TEST_END   = '2014-12-31'
H          = 24   # horizonte
INPUT_SIZE = 168  # 7 días de historia

panel_trainval = panel.loc[:VAL_END]
panel_test     = panel.loc[TEST_START:]
y_train_arr    = panel.loc[:TRAIN_END].values.T  # (N, T_train)

print(f'Train+Val: {panel_trainval.index.min()} → {panel_trainval.index.max()} ({len(panel_trainval)} pasos)')
print(f'Test     : {panel_test.index.min()} → {panel_test.index.max()} ({len(panel_test)} pasos)')

In [ ]:
# ── Convertir panel wide → formato largo Nixtla (unique_id, ds, y) ───────────
def panel_to_nixtla(df: pd.DataFrame) -> pd.DataFrame:
    """Convierte un DataFrame wide (índice=timestamp, cols=clientes) a formato largo Nixtla."""
    long = df.reset_index().melt(id_vars=df.index.name or 'index',
                                  var_name='unique_id', value_name='y')
    long = long.rename(columns={df.index.name or 'index': 'ds'})
    long['ds'] = pd.to_datetime(long['ds'])
    long = long[['unique_id', 'ds', 'y']].sort_values(['unique_id', 'ds']).reset_index(drop=True)
    long['y'] = long['y'].fillna(0.0)
    return long

df_nixtla_trainval = panel_to_nixtla(panel_trainval)
print(f'Formato Nixtla train+val: {df_nixtla_trainval.shape}')
print(df_nixtla_trainval.head(3))

## 2. Definición de Modelos

In [ ]:
# ── N-HiTS ────────────────────────────────────────────────────────────────────
# Neural Hierarchical Interpolation for Time Series Forecasting (Challu et al., 2022)
# Descompone el horizonte jerárquicamente para capturar patrones a distintas escalas.

model_nhits = NHITS(
    h=H,                        # horizonte de pronóstico
    input_size=INPUT_SIZE,      # ventana de historia (168 h = 7 días)
    max_steps=500,              # épocas de entrenamiento
    learning_rate=1e-3,
    batch_size=32,
    # Arquitectura
    n_stacks=3,                 # bloques jerárquicos
    n_blocks=[1, 1, 1],         # bloques por stack
    mlp_units=[[512, 512],      # capas MLP por bloque
               [512, 512],
               [512, 512]],
    n_pool_kernel_size=[4, 2, 1],  # pooling jerárquico
    n_freq_downsample=[8, 4, 1],   # downsample de frecuencia
    random_seed=42,
    early_stop_patience_steps=10,
    val_check_steps=50,
    scaler_type='standard',
)

# ── TCN ───────────────────────────────────────────────────────────────────────
# Temporal Convolutional Network con dilated causal convolutions.

model_tcn = TCN(
    h=H,
    input_size=INPUT_SIZE,
    max_steps=500,
    learning_rate=1e-3,
    batch_size=32,
    # Arquitectura
    encoder_hidden_size=128,    # canales por capa convolucional
    context_size=10,            # contexto de atención (si aplica)
    random_seed=42,
    early_stop_patience_steps=10,
    val_check_steps=50,
    scaler_type='standard',
)

print('Modelos definidos:')
print(f'  N-HiTS → {model_nhits}')
print(f'  TCN    → {model_tcn}')

## 3. Entrenamiento (Train + Val)

In [ ]:
# ── Entrenar N-HiTS ───────────────────────────────────────────────────────────
print('Entrenando N-HiTS...')
nf_nhits = NeuralForecast(models=[model_nhits], freq='H')
nf_nhits.fit(df=df_nixtla_trainval, val_size=H*30)  # ~30 días como validación
print('N-HiTS entrenado ✓')

In [ ]:
# ── Entrenar TCN ──────────────────────────────────────────────────────────────
print('Entrenando TCN...')
nf_tcn = NeuralForecast(models=[model_tcn], freq='H')
nf_tcn.fit(df=df_nixtla_trainval, val_size=H*30)
print('TCN entrenado ✓')

## 4. Wrapper `NeuralForecastModel` (Protocolo `ForecastModel`)

In [ ]:
class NeuralForecastModel:
    """
    Wrapper que adapta un objeto NeuralForecast ya entrenado al protocolo
    ForecastModel de src/backtest.py.

    ForecastModel.predict(history, horizon, origin_time) → ndarray (N, H)
    """

    def __init__(self, nf: NeuralForecast, model_name: str, clients: list,
                 input_size: int = 168):
        """
        Args:
            nf         : NeuralForecast entrenado.
            model_name : nombre legible del modelo (e.g. 'NHITS').
            clients    : lista de unique_ids (columnas del panel).
            input_size : ventana de contexto (debe coincidir con el modelo).
        """
        self.nf         = nf
        self._name      = model_name
        self.clients    = list(clients)
        self.input_size = input_size

    # ── Protocolo ForecastModel ───────────────────────────────────────────────

    @property
    def name(self) -> str:
        return self._name

    def predict(
        self,
        history: pd.DataFrame,
        horizon: int,
        origin_time: pd.Timestamp,
    ) -> np.ndarray:
        """
        Genera pronósticos para las `horizon` horas siguientes.

        Args:
            history     : DataFrame (T x N) en escala original.
            horizon     : pasos a pronosticar (24).
            origin_time : instante de emisión del pronóstico.

        Returns:
            array (N, horizon).
        """
        # Recortar la ventana de contexto requerida por el modelo
        ctx = history.iloc[-self.input_size:]

        # Construir DataFrame largo Nixtla con la historia disponible
        df_ctx = ctx.reset_index().melt(
            id_vars=[ctx.index.name or 'index'],
            var_name='unique_id',
            value_name='y',
        ).rename(columns={ctx.index.name or 'index': 'ds'})
        df_ctx['ds'] = pd.to_datetime(df_ctx['ds'])
        df_ctx['y']  = df_ctx['y'].fillna(0.0)
        df_ctx = df_ctx[['unique_id', 'ds', 'y']].sort_values(['unique_id', 'ds'])

        # Pronóstico
        preds_df = self.nf.predict(df=df_ctx)

        # La columna de predicciones se llama como el modelo (p.ej. 'NHITS')
        pred_col = [c for c in preds_df.columns if c not in ('unique_id', 'ds')][0]

        # Reorganizar a shape (N, H): una fila por cliente
        preds_pivot = (
            preds_df
            .sort_values(['unique_id', 'ds'])
            .pivot(index='unique_id', columns='ds', values=pred_col)
            .reindex(self.clients)           # orden canónico de clientes
            .fillna(0.0)
        )
        return preds_pivot.values  # (N, H)


print('Wrapper NeuralForecastModel definido ✓')

## 5. Backtest Rolling-Origin en Prueba

In [ ]:
clients = list(panel.columns)

# Instanciar wrappers
wrapped_nhits = NeuralForecastModel(nf_nhits, 'NHITS', clients, input_size=INPUT_SIZE)
wrapped_tcn   = NeuralForecastModel(nf_tcn,   'TCN',   clients, input_size=INPUT_SIZE)

print('Wrappers instanciados ✓')
print(f'  Modelo NHITS: {wrapped_nhits.name}')
print(f'  Modelo TCN  : {wrapped_tcn.name}')

In [ ]:
# ── Backtest N-HiTS ───────────────────────────────────────────────────────────
print('Ejecutando backtest N-HiTS (92 orígenes)...')
result_nhits = backtest(
    model=wrapped_nhits,
    panel=panel,
    y_train=y_train_arr,
    H=H,
    test_start=TEST_START,
    test_end=TEST_END,
    seasonality=24,
    verbose=True,
    return_predictions=True,
)
print('Backtest N-HiTS completado ✓')
print(pd.Series(result_nhits['metrics']))

In [ ]:
# ── Backtest TCN ──────────────────────────────────────────────────────────────
print('Ejecutando backtest TCN (92 orígenes)...')
result_tcn = backtest(
    model=wrapped_tcn,
    panel=panel,
    y_train=y_train_arr,
    H=H,
    test_start=TEST_START,
    test_end=TEST_END,
    seasonality=24,
    verbose=True,
    return_predictions=True,
)
print('Backtest TCN completado ✓')
print(pd.Series(result_tcn['metrics']))

## 6. Métricas de Prueba

In [ ]:
metrics_nhits = result_nhits['metrics']
metrics_tcn   = result_tcn['metrics']

metrics_df = pd.DataFrame([
    {k: v for k, v in metrics_nhits.items()},
    {k: v for k, v in metrics_tcn.items()},
])

# Formatear tabla
display_cols = ['model', 'MAE', 'RMSE', 'WAPE', 'MASE', 'inference_time_s']
display_cols = [c for c in display_cols if c in metrics_df.columns]
print('\n📊 Métricas de prueba:')
print(metrics_df[display_cols].to_string(index=False))

## 7. Guardar Predicciones

In [ ]:
OUTPUTS_DIR = os.path.join(ROOT, 'outputs')
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Concatenar predicciones de ambos modelos
preds_all = pd.concat([
    result_nhits['predictions'],
    result_tcn['predictions'],
], ignore_index=True)

out_path = os.path.join(OUTPUTS_DIR, 'predictions_nhits_tcn.csv')
preds_all.to_csv(out_path, index=False)
print(f'Predicciones guardadas en: {out_path}')
print(f'Shape: {preds_all.shape}')
print(preds_all.head(3))

## 8. MAE por Paso del Horizonte

In [ ]:
mae_h_nhits = result_nhits['mae_horizon']  # array (H,)
mae_h_tcn   = result_tcn['mae_horizon']

fig, ax = plt.subplots(figsize=(12, 5))

horizons = np.arange(1, H + 1)

ax.plot(horizons, mae_h_nhits, marker='o', markersize=4, linewidth=2,
        color='royalblue', label='N-HiTS')
ax.plot(horizons, mae_h_tcn,   marker='s', markersize=4, linewidth=2,
        color='tomato',     label='TCN')

ax.set_xlabel('Paso del horizonte (h)', fontsize=12)
ax.set_ylabel('MAE (kWh)', fontsize=12)
ax.set_title('MAE por Paso del Horizonte — N-HiTS vs TCN', fontsize=14, fontweight='bold')
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)

plt.tight_layout()
fig_path = os.path.join(ROOT, 'outputs', 'figures', 'mae_horizon_nhits_tcn.png')
os.makedirs(os.path.dirname(fig_path), exist_ok=True)
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada: {fig_path}')

## 9. Resumen Final

In [ ]:
print('=' * 60)
print('RESUMEN — Sección 5: N-HiTS y TCN')
print('=' * 60)
for name, metrics in [('N-HiTS', metrics_nhits), ('TCN', metrics_tcn)]:
    print(f'\n{name}:')
    for k in ['MAE', 'RMSE', 'WAPE', 'MASE']:
        if k in metrics:
            print(f'  {k:10s}: {metrics[k]:.4f}')
print(f'\nPredicciones → outputs/predictions_nhits_tcn.csv')
print(f'Figura       → outputs/figures/mae_horizon_nhits_tcn.png')